# Phase 1 - Historical ETL validation

Sanity-checks the BigQuery marts produced by `etl.run_full` against the
expectations called out in the Phase 1 runbook and the project plan.

Run from the activated `.venv` with `BQ` credentials in place.

Outputs the same numbers the agent will rely on later, so any mismatch
here predicts a Phase 5 reasoning failure.

In [ ]:
from etl.config import settings
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=settings.gcp_project_id, location=settings.gcp_bq_location)
marts = f"{settings.gcp_project_id}.{settings.bq_dataset_marts}"
print("querying:", marts)

## 1. Season coverage

In [ ]:
sql = f"""
SELECT season, COUNT(*) AS n
FROM `{marts}.match`
GROUP BY 1
ORDER BY 1
"""
df = client.query(sql).to_dataframe()
print(f"seasons in scope: {len(df)}")
assert len(df) >= 14, "expected ~16 EPL seasons from OpenFootball (2010-11 onward)"
df.tail()

## 2. xG-enriched fixtures (Understat join)

In [ ]:
sql = f"""
SELECT season, home_team, away_team, home_xg, away_xg
FROM `{marts}.match`
WHERE home_xg IS NOT NULL
ORDER BY season DESC, match_date DESC
LIMIT 20
"""
df = client.query(sql).to_dataframe()
assert len(df) > 0, "expected Understat xG on the match mart"
df

## 3. Team head-to-head (OpenFootball)

In [ ]:
sql = f"""
SELECT team_a, team_b, played, wins, goals_for, goals_against
FROM `{marts}.head_to_head`
WHERE team_a = 'Chelsea' OR team_b = 'Chelsea'
ORDER BY played DESC
LIMIT 10
"""
df = client.query(sql).to_dataframe()
df